In [ ]:
from pathlib import Path
import subprocess, numpy as np, pandas as pd

IN_PKL     = "/Users/trinityrose/Desktop/acs_ssc_predicted_fixed.pkl"
OUT_PKL    = "/Users/trinityrose/Desktop/acs_ssc_msub_v2.pkl"
TAXSIM_EXE = "/Users/trinityrose/taxsim35"   

df = pd.read_pickle(IN_PKL)
print(f"Loaded: {df.shape}")

In [ ]:
# FIPS → TAXSIM SOI state code (alphabetical, 1-51, DC=9)
FIPS_TO_SOI = {
    1:1,  2:2,  4:3,  5:4,  6:5,  8:6,  9:7,  10:8,  11:9,  12:10,
    13:11, 15:12, 16:13, 17:14, 18:15, 19:16, 20:17, 21:18, 22:19, 23:20,
    24:21, 25:22, 26:23, 27:24, 28:25, 29:26, 30:27, 31:28, 32:29, 33:30,
    34:31, 35:32, 36:33, 37:34, 38:35, 39:36, 40:37, 41:38, 42:39, 44:40,
    45:41, 46:42, 47:43, 48:44, 49:45, 50:46, 51:47, 53:48, 54:49, 55:50, 56:51
}

def build_taxsim_input(df, earn_r_col, earn_sp_col):
    rows = []
    for i, row in enumerate(df.itertuples(index=False)):
        earn_r  = max(float(getattr(row, earn_r_col,  0) or 0), 0)
        earn_sp = max(float(getattr(row, earn_sp_col, 0) or 0), 0)
        kids    = int(row.c_children)
        soi     = FIPS_TO_SOI.get(int(row.statefip), 0)
        year    = int(row.year)
        age_r   = int(row.age)
        age_sp  = int(row.sp_age)

        if earn_r >= earn_sp:
            hi, lo, age_hi, age_lo = earn_r, earn_sp, age_r, age_sp
            kids_hi, kids_lo = kids, 0
        else:
            hi, lo, age_hi, age_lo = earn_sp, earn_r, age_sp, age_r
            kids_hi, kids_lo = 0, kids

        # Row 0: higher earner single
        rows.append({"taxsimid": i*3+1, "year": year, "state": soi,
                     "mstat": 1, "page": age_hi, "sage": 0,
                     "depx": kids_hi, "pwages": hi, "swages": 0})
        # Row 1: lower earner single
        rows.append({"taxsimid": i*3+2, "year": year, "state": soi,
                     "mstat": 1, "page": age_lo, "sage": 0,
                     "depx": kids_lo, "pwages": lo, "swages": 0})
        # Row 2: couple joint
        rows.append({"taxsimid": i*3+3, "year": year, "state": soi,
                     "mstat": 2, "page": age_hi, "sage": age_lo,
                     "depx": kids, "pwages": hi, "swages": lo})

    return pd.DataFrame(rows)


def run_taxsim(taxsim_input_df, exe_path):
    csv_str = taxsim_input_df.to_csv(index=False)
    result  = subprocess.run(
        [str(exe_path)],
        input=csv_str, capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"TAXSIM error:\n{result.stderr}")
    from io import StringIO
    return pd.read_csv(StringIO(result.stdout))


def compute_subsidies_taxsim(df, earn_r_col, earn_sp_col, suffix, exe_path):
    print(f"  Building TAXSIM input ({suffix})...")
    inp = build_taxsim_input(df, earn_r_col, earn_sp_col)
    print(f"  Sending {len(inp):,} rows to TAXSIM...")
    out = run_taxsim(inp, exe_path)

    out = out.set_index("taxsimid")[["fiitax", "siitax"]]
    n   = len(df)

    t_hi_fed = out.loc[range(1, n*3+1, 3), "fiitax"].values
    t_lo_fed = out.loc[range(2, n*3+1, 3), "fiitax"].values
    t_jt_fed = out.loc[range(3, n*3+1, 3), "fiitax"].values
    t_hi_st  = out.loc[range(1, n*3+1, 3), "siitax"].values
    t_lo_st  = out.loc[range(2, n*3+1, 3), "siitax"].values
    t_jt_st  = out.loc[range(3, n*3+1, 3), "siitax"].values

    msub_fed   = (t_hi_fed + t_lo_fed) - t_jt_fed
    msub_state = (t_hi_st  + t_lo_st)  - t_jt_st

    # Zero state subsidy where state doesn't recognise the marriage
    no_state_recog = df["staterecog_policy"].values == 0
    msub_state[no_state_recog] = 0.0

    # Zero everything pre-Windsor (year <= 2012)
    pre_windsor = df["preW"].values == 1
    msub_fed[pre_windsor]   = 0.0
    msub_state[pre_windsor] = 0.0

    msub_total = msub_fed + msub_state

    out_df = df.copy()
    out_df[f"msub_fed_{suffix}"]   = msub_fed
    out_df[f"msub_state_{suffix}"] = msub_state
    out_df[f"msub_total_{suffix}"] = msub_total
    return out_df

print("Functions defined.")

In [ ]:
# ── Run observed ─────────────────────────────────────────────────
print("Observed subsidy (reported earnings)...")
df = compute_subsidies_taxsim(df, "r_incearn", "sp_incearn", "obs", TAXSIM_EXE)

In [ ]:
print(IN_PKL)
print(df.columns.tolist())

In [ ]:
# ── Run predicted (instrument) ───────────────────────────────────
print("Predicted subsidy (Lasso earnings)...")
df = compute_subsidies_taxsim(df, "hat_incearn_r", "hat_incearn_sp", "hat", TAXSIM_EXE)

In [ ]:
# ── Diagnostics ──────────────────────────────────────────────────
for label, mask in [("Married", df["sscouple_mar"]), ("Cohabiting", df["sscouple_coh"])]:
    sub = df[mask]
    print(f"\n{label} (N={mask.sum():,}):")
    print(f"  Fed+state observed:  {sub['msub_total_obs'].mean():8.1f}  (paper: mar=442  coh=264)")
    print(f"  Fed+state predicted: {sub['msub_total_hat'].mean():8.1f}  (paper: mar=68   coh=257)")
    print(f"  Federal observed:    {sub['msub_fed_obs'].mean():8.1f}  (paper: mar=395  coh=232)")
    print(f"  State observed:      {sub['msub_state_obs'].mean():8.1f}  (paper: mar=47   coh=32)")

In [ ]:
# ── Save ─────────────────────────────────────────────────────────
df.to_pickle(OUT_PKL)
print(f"Saved to {OUT_PKL}")